In [5]:
# ============================================
# COMPLETE CODE - Copy paste karo ye sab
# ============================================

# Install all packages
!pip install -q nltk pandas numpy scikit-learn PyPDF2 docx2txt streamlit matplotlib seaborn plotly

# Imports
import streamlit as st
import pandas as pd
import numpy as np
import re
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from PyPDF2 import PdfReader
import docx2txt
import plotly.graph_objects as go
from datetime import datetime
import io

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ============================================
# ResumeMatchNet Class
# ============================================
class ResumeMatchNet:
    def __init__(self):
        self.text_weight = 0.6
        self.skill_weight = 0.4
        self.vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
        
    def preprocess_text(self, text):
        text = text.lower()
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        words = nltk.word_tokenize(text)
        stop_words = set(stopwords.words('english'))
        words = [w for w in words if w not in stop_words and len(w) > 2]
        return ' '.join(words)
    
    def extract_skills(self, text):
        skill_keywords = [
            'python', 'java', 'javascript', 'sql', 'nosql', 'mongodb', 'postgresql',
            'mysql', 'django', 'flask', 'react', 'angular', 'vue', 'node.js', 'express',
            'tensorflow', 'pytorch', 'scikit-learn', 'pandas', 'numpy', 'matplotlib',
            'aws', 'azure', 'gcp', 'docker', 'kubernetes', 'jenkins', 'git', 'github',
            'rest', 'api', 'microservices', 'html', 'css', 'bootstrap', 'tailwind',
            'machine learning', 'deep learning', 'nlp', 'computer vision', 'data analysis',
            'excel', 'tableau', 'power bi', 'hadoop', 'spark', 'kafka', 'redis', 'rabbitmq'
        ]
        text_lower = text.lower()
        found_skills = set()
        for skill in skill_keywords:
            if skill in text_lower:
                found_skills.add(skill)
        return found_skills
    
    def calculate_text_similarity(self, resume, job):
        resume_processed = self.preprocess_text(resume)
        job_processed = self.preprocess_text(job)
        
        if not resume_processed or not job_processed:
            return 0.0
        
        try:
            tfidf_matrix = self.vectorizer.fit_transform([resume_processed, job_processed])
            similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
            return round(similarity[0][0] * 100, 2)
        except:
            return 0.0
    
    def calculate_skill_match(self, resume_skills, job_skills):
        if not job_skills:
            return 100.0, {}
        
        matched = resume_skills.intersection(job_skills)
        missing = job_skills - resume_skills
        
        score = (len(matched) / len(job_skills)) * 100 if job_skills else 0
        
        return round(score, 2), {
            'matched': len(matched),
            'missing': len(missing),
            'job': len(job_skills),
            'resume': len(resume_skills)
        }
    
    def match(self, resume_text, job_text):
        resume_skills = self.extract_skills(resume_text)
        job_skills = self.extract_skills(job_text)
        
        text_sim = self.calculate_text_similarity(resume_text, job_text)
        skill_score, skill_count = self.calculate_skill_match(resume_skills, job_skills)
        
        final_score = (text_sim * self.text_weight) + (skill_score * self.skill_weight)
        
        return {
            'match_score': round(final_score, 2),
            'text_similarity': text_sim,
            'skill_match_score': skill_score,
            'resume_skills': list(resume_skills),
            'job_skills': list(job_skills),
            'matched_skills': list(resume_skills.intersection(job_skills)),
            'missing_skills': list(job_skills - resume_skills),
            'skill_count': skill_count
        }

# ============================================
# Helper Functions
# ============================================
def process_uploaded_file(uploaded_file):
    try:
        file_extension = uploaded_file.name.split('.')[-1].lower()
        
        if file_extension == 'pdf':
            pdf_reader = PdfReader(uploaded_file)
            text = ''
            for page in pdf_reader.pages:
                text += page.extract_text()
            return text
        
        elif file_extension == 'docx':
            text = docx2txt.process(uploaded_file)
            return text
        
        elif file_extension == 'txt':
            text = uploaded_file.read().decode('utf-8')
            return text
        
        else:
            return "Error: Unsupported file format"
    
    except Exception as e:
        return f"Error processing file: {str(e)}"

def get_all_job_titles():
    return [
        "Senior Python Developer",
        "Data Scientist",
        "Machine Learning Engineer",
        "Full Stack Developer",
        "Frontend Developer",
        "Backend Developer",
        "DevOps Engineer",
        "Data Analyst"
    ]

def get_job_description(job_title):
    jobs_db = {
        "Senior Python Developer": """
        We are looking for a Senior Python Developer with 5+ years of experience.
        Required Skills: Python, Django, Flask, SQL, MongoDB, REST APIs, Git, Docker
        Experience with AWS or cloud platforms is a plus.
        """,
        
        "Data Scientist": """
        Data Scientist needed for our analytics team.
        Required Skills: Python, pandas, numpy, scikit-learn, SQL, Machine Learning, Statistics
        Experience with TensorFlow or PyTorch preferred.
        """,
        
        "Machine Learning Engineer": """
        ML Engineer to build and deploy models.
        Required Skills: Python, TensorFlow, PyTorch, scikit-learn, Docker, Kubernetes, AWS
        Strong background in ML algorithms and MLOps.
        """,
        
        "Full Stack Developer": """
        Full Stack Developer proficient in modern web technologies.
        Required Skills: JavaScript, React, Node.js, Python, Django, SQL, MongoDB, Git, REST APIs
        """,
        
        "Frontend Developer": """
        Frontend Developer with strong UI/UX skills.
        Required Skills: JavaScript, React, HTML5, CSS3, Tailwind CSS, Git, REST APIs
        """,
        
        "Backend Developer": """
        Backend Developer to build scalable APIs.
        Required Skills: Python, Django, Flask, SQL, MongoDB, Redis, RabbitMQ, Docker, Git
        """,
        
        "DevOps Engineer": """
        DevOps Engineer for cloud infrastructure.
        Required Skills: Docker, Kubernetes, AWS, Jenkins, Git, Linux, Python, Terraform
        """,
        
        "Data Analyst": """
        Data Analyst to derive insights from data.
        Required Skills: SQL, Python, pandas, Excel, Tableau, Power BI, Statistics
        """
    }
    return jobs_db.get(job_title, "Job description not found.")

# ============================================
# Streamlit UI
# ============================================
def main():
    st.set_page_config(
        page_title="ResumeMatchNet - Smart Resume Matcher",
        page_icon="🎯",
        layout="wide"
    )
    
    st.markdown("""
    <style>
        .main-header {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 2rem;
            border-radius: 10px;
            color: white;
            text-align: center;
        }
        .score-card {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 1.5rem;
            border-radius: 15px;
            color: white;
            text-align: center;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }
        .skill-badge {
            background-color: #e0e7ff;
            padding: 0.3rem 0.8rem;
            border-radius: 20px;
            font-size: 0.8rem;
            margin: 0.2rem;
            display: inline-block;
        }
        .metric-card {
            background-color: #f8f9fa;
            padding: 1rem;
            border-radius: 10px;
            text-align: center;
            border-left: 4px solid #667eea;
        }
    </style>
    """, unsafe_allow_html=True)
    
    st.markdown("""
    <div class="main-header">
        <h1>🎯 ResumeMatchNet</h1>
        <p>AI-Powered Resume-Job Description Matching System</p>
    </div>
    """, unsafe_allow_html=True)
    
    st.markdown("---")
    
    with st.sidebar:
        st.header("⚙️ Settings")
        text_weight = st.slider("Text Similarity Weight", 0.0, 1.0, 0.6, 0.1)
        skill_weight = st.slider("Skills Match Weight", 0.0, 1.0, 0.4, 0.1)
        st.markdown("---")
        st.info("💡 **Tip:** Upload your resume and select a job to get matching score!")
    
    col1, col2 = st.columns(2)
    
    with col1:
        st.header("📄 Upload Your Resume")
        upload_option = st.radio("Choose input method:", ["Upload File", "Paste Text"])
        resume_text = ""
        
        if upload_option == "Upload File":
            uploaded_file = st.file_uploader("Choose file (PDF, DOCX, or TXT)", type=['pdf', 'docx', 'txt'])
            if uploaded_file:
                resume_text = process_uploaded_file(uploaded_file)
                if resume_text and not "Error" in resume_text:
                    st.success("✅ File processed!")
        else:
            resume_text = st.text_area("Paste your resume:", height=200)
    
    with col2:
        st.header("💼 Job Description")
        job_option = st.radio("Choose job input:", ["Select from Database", "Paste Job Description"])
        job_text = ""
        
        if job_option == "Select from Database":
            job_titles = get_all_job_titles()
            selected_job = st.selectbox("Select job:", job_titles)
            if selected_job:
                job_text = get_job_description(selected_job)
                st.text_area("Job Description:", job_text, height=150)
        else:
            job_text = st.text_area("Paste job description:", height=200)
    
    if st.button("🔍 MATCH RESUME WITH JOB", type="primary", use_container_width=True):
        if not resume_text:
            st.error("❌ Please provide a resume")
        elif not job_text:
            st.error("❌ Please provide a job description")
        else:
            model = ResumeMatchNet()
            model.text_weight = text_weight
            model.skill_weight = skill_weight
            result = model.match(resume_text, job_text)
            
            st.markdown("---")
            st.header("📊 Matching Results")
            
            col1, col2, col3 = st.columns(3)
            with col1:
                st.markdown(f"""
                <div class="score-card">
                    <h3>Overall Match</h3>
                    <h1 style="font-size: 3rem;">{result['match_score']}%</h1>
                </div>
                """, unsafe_allow_html=True)
            with col2:
                st.markdown(f"""
                <div class="metric-card">
                    <h4>🎯 Skills Match</h4>
                    <h2>{result['skill_match_score']}%</h2>
                </div>
                """, unsafe_allow_html=True)
            with col3:
                st.markdown(f"""
                <div class="metric-card">
                    <h4>📝 Text Similarity</h4>
                    <h2>{result['text_similarity']}%</h2>
                </div>
                """, unsafe_allow_html=True)
            
            st.subheader("✅ Matched Skills")
            if result['matched_skills']:
                st.write(", ".join(result['matched_skills'][:10]))
            
            st.subheader("❌ Missing Skills")
            if result['missing_skills']:
                st.write(", ".join(result['missing_skills'][:10]))

if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
2026-04-02 23:14:19.315 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-02 23:14:19.316 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-02 23:14:19.316 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-02 23:14:19.316 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-02 23:14:19.317 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-02 23:14:19.317 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-02 23:14:19.318 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare